<a href="https://colab.research.google.com/github/Aditya-V-Naidoo/UoY_NLP_practical/blob/main/UoY_NLPG_Practical_1_Shannon_Markov_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLPG: Natural Language Processing
## Practical 1A — Shannon-style sequence models

**University of York**

In Lecture 1 we treated language as an observable sequence generated by some underlying model. In this practical we make that idea concrete by building simple **finite-order / Markov language models** at two tokenisation levels:

- **character level**: characters, spaces and punctuation form the sequence;
- **word level**: word and punctuation tokens form the sequence.

You will train models from **two different public-domain books**, vary how much previous context the model remembers, and generate samples.

### During this practical

There is **nothing to submit**, and you do not need to edit this notebook to record answers. As you work, **take your own notes** on the required activities and what you observe.

Keep your notes and bring them to this week's summing-up lecture (L2), **where we may compare and discuss your findings**.

There are **three required activities**. Material marked **Further exploration (optional)** is there if you have time or want to investigate further.

> The generated output is **evidence to analyse**, not just something to obtain.


## 1. From Shannon's experiment to our notebook

Shannon's famous "successive approximations" to English used progressively more information about what had already appeared in a sequence. One way to recreate the same basic idea computationally is to estimate probabilities from a corpus.

The key modelling choice is: **what counts as one symbol?**

Consider:

> `the cat sat`

At **character level**, a possible token sequence is:

`['t', 'h', 'e', ' ', 'c', 'a', 't', ' ', 's', 'a', 't']`

At **word level**, a possible token sequence is:

`['the', 'cat', 'sat']`

The same visible text therefore becomes a very different statistical sequence depending on the tokenisation.

### Context length and model order

We will describe the model mainly by its **context length** \(k\):

\[
P(x_t \mid x_{t-k}, \ldots, x_{t-1})
\]

- \(k=0\): choose the next token without looking back — a **unigram** model;
- \(k=1\): look at one previous token — a **bigram** model;
- \(k=2\): look at two previous tokens — a **trigram** model;
- and so on.

NLTK conventionally calls the corresponding \(n\)-gram model's order \(n=k+1\).

| Context remembered | N-gram name | Model order |
|---:|---|---:|
| 0 tokens | unigram | 1 |
| 1 token | bigram | 2 |
| 2 tokens | trigram | 3 |
| 4 tokens | 5-gram | 5 |

At character level, a context of 4 characters is still quite local.  
At word level, a context of 4 words is already much more specific and therefore much sparser.

## 2. Setup

We use **NLTK's Gutenberg corpus** rather than downloading books manually. The texts are public-domain books distributed with NLTK, which makes this notebook easy to run in Colab.

Our two corpora will be:

- Lewis Carroll, *Alice's Adventures in Wonderland*
- Herman Melville, *Moby Dick*

They are deliberately different in subject matter and style.

You can later replace either corpus with another Gutenberg text, or with another plain-text public-domain source.

In [ ]:
import nltk
import random
import re
import pandas as pd

from nltk.corpus import gutenberg
from nltk.tokenize import TreebankWordTokenizer, TreebankWordDetokenizer
from nltk.lm import MLE
from nltk.util import everygrams

nltk.download("gutenberg", quiet=True)

print("NLTK version:", nltk.__version__)
print("Available Gutenberg files:")
print(gutenberg.fileids())

NLTK version: 3.9.1
Available Gutenberg files:
['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']


## 3. Load and inspect the two corpora

In [ ]:
CORPORA = {
    "Alice": "carroll-alice.txt",
    "Moby Dick": "melville-moby_dick.txt",
}

raw_texts = {name: gutenberg.raw(fileid) for name, fileid in CORPORA.items()}

for name, text in raw_texts.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(f"{len(text):,} characters")
    print(text[:700].replace("\n", " "))


Alice
-----
144,395 characters
[Alice's Adventures in Wonderland by Lewis Carroll 1865]  CHAPTER I. Down the Rabbit-Hole  Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, 'and what is the use of a book,' thought Alice 'without pictures or conversation?'  So she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting up and picking the daisies, when suddenly a White Rabbit with pink eyes ran close by her.  There was nothi

Moby Dick
---------
1,242,990 characters
 of the word, you deliver that which is not t


## 4. Normalisation is already a modelling decision

We will do only light preprocessing:

1. convert runs of whitespace (`\n`, tabs, repeated spaces) to a single space;
2. optionally lowercase the text.

Lowercasing reduces sparsity, but it also removes information. There is no universally "correct" choice: preprocessing changes the data presented to the model.

For the main experiment we lowercase both corpora so that case differences do not dominate the comparison.

In [ ]:
def normalise_text(text, lowercase=True):
    # Collapse whitespace and optionally lowercase.
    text = re.sub(r"\s+", " ", text).strip()
    if lowercase:
        text = text.lower()
    return text

texts = {name: normalise_text(text, lowercase=True)
         for name, text in raw_texts.items()}

for name, text in texts.items():
    print(name, ":", repr(text[:180]))

Alice : "[alice's adventures in wonderland by lewis carroll 1865] chapter i. down the rabbit-hole alice was beginning to get very tired of sitting by her sister on the bank, and of having n"
Moby Dick : '[moby dick by herman melville 1851] etymology. (supplied by a late consumptive usher to a grammar school) the pale usher--threadbare in coat, heart, body, and brain; i see him now.'


# REQUIRED 1/3 — Characters and words are different representations

We now turn each text into two different sequences.

### 5.1 Character tokens

At character level, **spaces and punctuation are tokens too**.  
This is important: without spaces, the model would have no direct evidence about word boundaries.

### 5.2 Word tokens

For word-level modelling we use NLTK's `TreebankWordTokenizer`. It separates most punctuation and handles many English contractions in a consistent way.

There are many other valid tokenisers. Choosing one changes the sequence and therefore changes the model.

In [ ]:
word_tokenizer = TreebankWordTokenizer()
detokenizer = TreebankWordDetokenizer()

def char_tokens(text):
    return list(text)

def word_tokens(text):
    return word_tokenizer.tokenize(text)

example = "The cat can't sleep, apparently."

print("Original:")
print(example)

print("\nCharacter tokens:")
print(char_tokens(example))

print("\nWord tokens:")
print(word_tokens(example.lower()))

Original:
The cat can't sleep, apparently.

Character tokens:
['T', 'h', 'e', ' ', 'c', 'a', 't', ' ', 'c', 'a', 'n', "'", 't', ' ', 's', 'l', 'e', 'e', 'p', ',', ' ', 'a', 'p', 'p', 'a', 'r', 'e', 'n', 't', 'l', 'y', '.']

Word tokens:
['the', 'cat', 'ca', "n't", 'sleep', ',', 'apparently', '.']


As you inspect the two representations, **make a note for yourself** of what changes and what you predict that might mean for a finite-order model. In particular, notice sequence length and the number/kinds of possible token types.

## 6. Prepare comparable training samples

The original books are not the same size. If one model sees much more training data than another, corpus size becomes another difference in our experiment.

For the first comparison, we therefore use the same maximum number of tokens from each corpus at each tokenisation level.

This is **not** the only sensible sampling strategy. Taking a prefix may itself introduce bias. Later you should think about how a different sampling procedure could change the result.

The limits below keep the notebook responsive in Colab while still providing enough data for useful models.

In [ ]:
MAX_CHAR_TOKENS = 120_000
MAX_WORD_TOKENS = 25_000

char_corpora = {
    name: char_tokens(text)[:MAX_CHAR_TOKENS]
    for name, text in texts.items()
}

word_corpora = {
    name: word_tokens(text)[:MAX_WORD_TOKENS]
    for name, text in texts.items()
}

summary_rows = []

for name in CORPORA:
    summary_rows.append({
        "corpus": name,
        "character tokens": len(char_corpora[name]),
        "character types": len(set(char_corpora[name])),
        "word tokens": len(word_corpora[name]),
        "word types": len(set(word_corpora[name])),
    })

pd.DataFrame(summary_rows)

,corpus,character tokens,character types,word tokens,word types
0,Alice,120000,46,25000,2723
1,Moby Dick,120000,51,25000,5157


## 7. Build a finite-order Markov / n-gram model

For a context length \(k\), the model estimates the next-token distribution from observed \(k\)-token contexts.

For example, with word context length \(k=2\), the model estimates quantities such as:

\[
P(\text{cat}\mid \text{the black})
\]

from corpus counts.

We use NLTK's **maximum-likelihood n-gram model (`MLE`)**. It is deliberately simple: no sophisticated smoothing, neural network, or semantic representation is involved.

This makes the experiment close to the core Shannon-style idea:

> **look at a finite amount of recent history and estimate what tends to come next.**

In [ ]:
def build_markov_model(tokens, context_length):
    # context_length k corresponds to n-gram order n = k + 1.
    if context_length < 0:
        raise ValueError("context_length must be >= 0")

    n = context_length + 1
    model = MLE(n)

    # Supply unigrams through n-grams from one long observed sequence.
    training_ngrams = everygrams(tokens, max_len=n)
    model.fit([training_ngrams], vocabulary_text=tokens)

    return model

### A tiny sanity check

Before using a whole book, verify that the machinery behaves sensibly on a sequence we understand.

In [ ]:
toy = list("abababababab")
toy_model = build_markov_model(toy, context_length=1)

print("After 'a':")
print(toy_model.context_counts(("a",)))

print("\nAfter 'b':")
print(toy_model.context_counts(("b",)))

After 'a':
<FreqDist with 1 samples and 6 outcomes>

After 'b':
<FreqDist with 1 samples and 5 outcomes>


You should see that after `a`, the model has observed `b`, and after `b`, it has observed `a`.

That is the finite-order assumption in its simplest form.

## 8. Generate from a model

Generation repeatedly does the following:

1. take the current context;
2. use the learned next-token distribution;
3. sample a next token;
4. append it;
5. update the context;
6. repeat.

A random seed makes our example reproducible. Changing the seed gives a different sample from the same model.

In [ ]:
def generate_tokens(model, num_tokens=100, seed=None, random_seed=42):
    # Generate tokens from a fitted NLTK language model.
    if seed is None:
        seed = []

    if model.order > 1:
        seed = list(seed)[-(model.order - 1):]
    else:
        seed = []

    generated = model.generate(
        num_words=num_tokens,
        text_seed=seed,
        random_seed=random_seed
    )

    if isinstance(generated, str):
        generated = [generated]

    return generated


def show_char_generation(model, n_chars=500, seed_text=" the ", random_seed=42):
    out = generate_tokens(
        model,
        n_chars,
        seed=list(seed_text),
        random_seed=random_seed
    )
    return "".join(out)


def show_word_generation(model, n_words=100, seed_words=("in", "the"), random_seed=42):
    out = generate_tokens(
        model,
        n_words,
        seed=list(seed_words),
        random_seed=random_seed
    )
    return detokenizer.detokenize(out)

# REQUIRED 2/3 — More context changes what the model can reproduce

Start with **Alice**. Increase the context length and inspect what happens.

Suggested contexts:

- 0 characters: no memory;
- 1 character;
- 2 characters;
- 4 characters;
- 8 characters.

Do not assume that "higher order" must always mean "better". Ask what kind of regularity each model can learn, and how much data is available for each context.

In [ ]:
CHAR_CONTEXTS = [0, 1, 2, 4, 8]

alice_char_models = {}

for k in CHAR_CONTEXTS:
    print("\n" + "=" * 78)
    print(f"ALICE — character context length k={k}  (n-gram order={k+1})")
    print("=" * 78)

    model = build_markov_model(char_corpora["Alice"], context_length=k)
    alice_char_models[k] = model

    print(show_char_generation(
        model,
        n_chars=500,
        seed_text=" the ",
        random_seed=7
    ))


ALICE — character context length k=0  (n-gram order=1)
c m hd h e  es ,luiew sa  bs ilei  'nebiearn?ihtoaw eo g npitbniiesufn nmysaen e   p aet eitstaedtv  -.gia eeiunhln trtree l  ' d    d tl add syfg  das  uh h hwtnad phrc,rysrso,hd  aanueuyud,,''ltsgmr ntrog rcrweeuo   tr swndi  wmhuets)aa:iae tdeitethhh e  r foichir iaaphiotelhhnehguntuaius  e : nrt on tw,uegys ehd'co ie clh yrw a ra etsa tin  ne ulr s tedita h.   'baoah d a oi!fu sehsehnxdsoled   oa  stna;ae eavwiawbd efh'h a e  a-ihonotecy om stlor hhsrsitnn-  d sillng rohhn oa ao'owgegnplm  a

ALICE — character context length k=1  (n-gram order=2)
g s nd m i' it chre,' t, aloche, a bonche thaneroit is l smerargeatthe shus, s k a t d t inout'alst be lie ilithein ussond r aice " al thaile tweabbe t 'whan myot haseraitutloued ' e sthowomaid quther stise tasulethad us turd a whesatof f ofadomeesathed id t lligedichaisedesapingandrmpt own an carps shatt wecut ind gh on ghe yow, 's! itof whe are whico thedof mace be t, be d se ly the

As you inspect the samples, **take notes on the progression**. Look for word-like fragments, complete words, punctuation/spacing patterns, locally plausible phrases, and errors that remain. Try to identify where particular kinds of regularity first become visible.

## One model, several samples

Before moving on, run the same trained model several times with different random seeds.

The model itself does not change. Only the particular sample generated from it changes.

As you look at the outputs, simply notice how much your impression of the model could depend on which sample you happened to see.

> **Observable data provides evidence about the model.**

One sample does not completely characterise that model.


In [ ]:
model = alice_char_models[4]

for random_seed in [1, 2, 3]:
    print(f"\n--- sample: random seed {random_seed} ---")
    print(show_char_generation(
        model,
        n_chars=400,
        seed_text=" the ",
        random_seed=random_seed
    ))



--- sample: random seed 1 ---
could nobody as it?' she had mad.' 'who you out the tried the poor little went with my jaws it?' said the got that protecting that a minute girl she waiting off; the greaturedly day to cut of executed to down anothings of the said to be a tried in a very so,' alice been way. 'i quite kid gloves, she fall sitting!' cried hard off with whiting upon a violence to asked. 'i began beds come once, and s

--- sample: random seed 2 ---
won't seemed took no and went on, of this, you, wonder withoughly think,' said them,' the ducheshire cat was exactly. 'i've behind the garden, as she cake, by was not, come been her flat, and not slowly, 'let's head, indeed, could shrieks, and shrink i canary and a twice, she roses?' forway; 'it might, who window?' said 'which was not much at all the end their own her long such daising it saw the 

--- sample: random seed 3 ---
door, and yet,' she same at ther shut a which she surprison,' the swam about as usurpatient angry for rep

## Further exploration (optional) — Same character models, different corpus

Now repeat selected context lengths for *Moby Dick*.

Use the **same context lengths, generated length, and random seed** so that corpus is the main thing we are changing.

In [ ]:
for corpus_name in ["Alice", "Moby Dick"]:
    print("\n\n" + "#" * 78)
    print(corpus_name.upper())
    print("#" * 78)

    for k in [1, 4, 8]:
        model = build_markov_model(char_corpora[corpus_name], context_length=k)

        print(f"\n--- character context k={k} ---")
        print(show_char_generation(
            model,
            n_chars=400,
            seed_text=" the ",
            random_seed=11
        ))



##############################################################################
ALICE
##############################################################################

--- character context k=1 ---
inthee mis f th ypor 'i' d kerelind ytot. f tereto crexe rof gsl, a t on s r id wigr ithay dg' abld, ither atof tt wngabawas the s chin.' ' i' he howhee ' vevarency ngrotly stonlopleeg ryoteneat t on'reit ' s atherar istone anoust' ie ind tun 't se t an t a relkithan 'de e seisast wichesanouthe sorofryseratom inin t t gl? moury'w chengere hit tthe nd, whereck ndst"-'buthe pt t benceicl, d me, main

--- character context k=4 ---
lory, and be so violent he other--anything as this eyes argue. i did, old, fathere was only the clearnt stalking very lifteen, and got it again, and, in a versation its mouse. 'very noise."' 'once of ther across, who have of rules and not turn the folded, 'and this minute or something pression, so she's six is stick, and converself, for, hasting of rude.' alice window.

# REQUIRED 3/3 — Changing the unit changes the modelling problem

At word level, each token carries much more information, but the vocabulary is dramatically larger.

That means a word model can produce recognisable lexical patterns with a short context, but exact multi-word contexts become sparse quickly.

We will begin with contexts \(k=0,1,2,3\).

Remember:

- word context \(k=0\) = unigram;
- word context \(k=1\) = bigram;
- word context \(k=2\) = trigram;
- word context \(k=3\) = 4-gram.

In [ ]:
WORD_CONTEXTS = [0, 1, 2, 3]

for k in WORD_CONTEXTS:
    print("\n" + "=" * 78)
    print(f"ALICE — word context length k={k}  (n-gram order={k+1})")
    print("=" * 78)

    model = build_markov_model(word_corpora["Alice"], context_length=k)

    print(show_word_generation(
        model,
        n_words=100,
        seed_words=("in", "the"),
        random_seed=21
    ))


ALICE — word context length k=0  (n-gram order=1)
, queen of him a tell the in if alice! different march'm than agony air.' your she thousand near' bright-eyed i, went desperately, the, very has kitchen cat hour:' think alice stirring a whole to she size mad see, herself the] us to find day happens down you drawing' 'yes said you i i key she an't lady no--, cheshire very chanced moment away, 'it the speak. do pool patriotic to what fellow you when melancholy to i do all through,'m the

ALICE — word context length k=1  (n-gram order=2)
distance, i do hope they weren't let me!' said, smiling at first, you my time she added in her; there ought not talk, when she ran off at any advantage of gloves in a well to see the mouse only alice could remember ever was the hearth and of life. the hatter's adventures in your majesty!' said alice hastily dried her as i can't had never forgotten that's the rabbit-hole alice looked like to twenty at you were nowhere to herself, "what a capital one

ALIC

Compare these outputs with your character-level samples. In your notes, concentrate on the trade-off created by the representation: what the character model has to learn, vocabulary size, sparsity at word level, and what you would — or would not — infer from locally grammatical-looking output.

## Further exploration (optional) — Compare the two corpora at word level

In [ ]:
results = []

for corpus_name in ["Alice", "Moby Dick"]:
    for k in [0, 1, 2]:
        model = build_markov_model(word_corpora[corpus_name], context_length=k)
        sample = show_word_generation(
            model,
            n_words=80,
            seed_words=("in", "the"),
            random_seed=31
        )

        results.append({
            "corpus": corpus_name,
            "context length": k,
            "order": k + 1,
            "generated sample": sample
        })

pd.DataFrame(results)

,corpus,context length,order,generated sample
0,Alice,0,1,"!, eggs poor,, against shouted, she only, it h..."
1,Alice,1,2,archbishop find herself from a consultation ab...
2,Alice,2,3,"act of crawling away: besides all this, she so..."
3,Moby Dick,0,1,"""; great proffer a; attitudes spile a sky own ..."
4,Moby Dick,1,2,alleghanian ridge in the cloth called bearskin...
5,Moby Dick,2,3,"afternoon of the ship's bluff bows, and that o..."


In [ ]:
# Edit this cell for your controlled experiment.

CORPUS_NAME = "Alice"          # "Alice" or "Moby Dick"
LEVEL = "char"                 # "char" or "word"
CONTEXT_LENGTH = 4
RANDOM_SEED = 123

if LEVEL == "char":
    tokens = char_corpora[CORPUS_NAME]
    model = build_markov_model(tokens, CONTEXT_LENGTH)
    output = show_char_generation(
        model,
        n_chars=700,
        seed_text=" the ",
        random_seed=RANDOM_SEED
    )
else:
    tokens = word_corpora[CORPUS_NAME]
    model = build_markov_model(tokens, CONTEXT_LENGTH)
    output = show_word_generation(
        model,
        n_words=130,
        seed_words=("in", "the"),
        random_seed=RANDOM_SEED
    )

print(output)

back-in-the e--even their hastill soon for the boots and picked dinary window, and look about as you, would not talk out at on, 'why,' but the mouse!' excelled alice, their head!"' 'solemn to seemed to company use open in since answer will you like a dear! done. alice fended anothing at the bring between her less by ever in the mock to tale (with is just be a low are you,' said to go do: one of the rattling. i could be little. it to be surprise could felt answer one about it, i've flowed, 'and the questing. 'a little had nothinking quite party said i'm neveral, dears, who i suppose. changing 'off being tone as to quite a low voice game,' said i don't goes insult gave sea. but it saw one of t


In [ ]:
model = build_markov_model(char_corpora["Alice"], context_length=4)

for seed in [1, 2, 3, 4]:
    print(f"\n--- random seed {seed} ---")
    print(show_char_generation(
        model,
        n_chars=300,
        seed_text=" the ",
        random_seed=seed
    ))


--- random seed 1 ---
could nobody as it?' she had mad.' 'who you out the tried the poor little went with my jaws it?' said the got that protecting that a minute girl she waiting off; the greaturedly day to cut of executed to down anothings of the said to be a tried in a very so,' alice been way. 'i quite kid gloves, she

--- random seed 2 ---
won't seemed took no and went on, of this, you, wonder withoughly think,' said them,' the ducheshire cat was exactly. 'i've behind the garden, as she cake, by was not, come been her flat, and not slowly, 'let's head, indeed, could shrieks, and shrink i canary and a twice, she roses?' forway; 'it mig

--- random seed 3 ---
door, and yet,' she same at ther shut a which she surprison,' the swam about as usurpatient angry for replied versation off to almost in my should. 'i'll thing!' alice. 'why direction at this sure i'll talking as it myself the same upon a minute offended to change in the jelly-fish i had silenty-fou

--- random seed 4 ---
decid

# Further exploration (optional)

If you have completed the three required activities, explore anything that interests you:

- compare *Alice* and *Moby Dick* while holding the other settings fixed;
- vary the amount of training data;
- preserve case rather than lowercasing;
- try another text from `gutenberg.fileids()`;
- keep a trained model fixed and generate many samples with different random seeds.

These are invitations to investigate, not additional required questions.


## What should you take away?

The amusing generated sentences are not the main result.

- **Tokenisation defines the sequence the model sees.**
- A finite-order model uses a bounded amount of recent context.
- More context can capture richer local regularities.
- Longer contexts also create sparsity, especially at word level.
- The corpus affects the distributions that are learned.
- Generated passages are samples from a model, not the model itself.
- Different generated samples can give different impressions of the same model.

In Lecture 1 we moved conceptually from

$$
\text{model} \rightarrow \text{observable sequence}
$$

to the usual NLP problem

$$
\text{observable data} \rightarrow \text{estimated model}.
$$

What we infer depends on **representation, corpus, finite samples and modelling assumptions**.

### Keep your notes

There is nothing to submit for these activities. Retain the notes you made while working and bring them to this week's summing-up lecture (L2), **where we may compare and discuss your findings**.
